In [1]:
# infer_parameters.py
from __future__ import annotations
from copy import deepcopy
from pathlib import Path
from typing import Sequence, Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.interpolate import interp1d

# Your utilities
from bioprocess_utils import (
    load_synthetic_dataset,        # load the CSV + meta (units, etc.)
    load_model_inputs,             # read inputs.json (params, initials, solver, t_span…)
    run_from_inputs,               # simulate
    fit_parameters
)

In [2]:
inputs = load_model_inputs("./inputs/inputs.json")
prms_to_infer = ("mu_max","K_subs","K_L_a")

In [3]:
display(inputs.keys())
display(inputs["mode_name"])

dict_keys(['params', 'initials', 'mode_name', 'mode_consts', 't_span', 't_eval', 'solver', 'meta'])

'fed-batch'

In [4]:

# Fit μmax, Ks, KLa from BIOMASS across all experiments in the CSV
report = fit_parameters(
    csv_path="./simulated_dataset/pputida_fedbatch_v1.wide.csv",
    inputs = inputs,
    observables=("Biomass","Sub"),
    param_names=prms_to_infer,
    initial_names=(),           # to also infer Biomass0: initial_names=("Biomass",)
    experiment_ids=None,        # None -> use all experiments; or pass e.g. [0]
    bounds=None,                # None -> auto ±10× around seeds
    x0=None,                    # None -> seeds from inputs.json
    verbose=2,
)
print("\n=== Fit summary ===")
for name, val in zip(report["theta_names"], report["theta_opt"]):
    print(f"{name:10s} = {val:.6g}")
print("success:", report["success"], "| cost:", report["cost"])

   Iteration     Total nfev        Cost      Cost reduction    Step norm     Optimality   
       0              1         2.2919e+00                                    8.96e+00    
       1              2         2.2549e+00      3.70e-02       3.26e-01       2.00e-02    
       2              3         2.2502e+00      4.66e-03       6.27e+00       2.43e-01    
       3              4         2.2465e+00      3.76e-03       3.91e-01       6.07e-03    
       4              5         2.2456e+00      8.80e-04       9.91e+00       2.25e-01    
       5             12         2.2456e+00      7.23e-09       4.09e-05       5.64e-02    
       6             13         2.2456e+00      2.61e-07       2.78e-06       1.07e-03    
       7             14         2.2456e+00      2.33e-08       2.83e-07       1.26e-03    
       8             15         2.2456e+00      5.22e-08       5.00e-07       1.74e-03    
       9             16         2.2456e+00      1.35e-08       1.18e-06       1.00e-02    